# Understanding TensorFlow by Building It from Scratch

We implement TensorFlow/Keras core concepts (GradientTape, Layers, Sequential, fit/compile) using only NumPy.
This demystifies the Keras API by showing what happens under the hood.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)


## 1. GradientTape: Recording Operations for Automatic Differentiation


In [ ]:
class GradientTape:
    """Records operations for reverse-mode autodiff, like tf.GradientTape."""
    
    def __init__(self):
        self.operations = []  # (op_name, inputs, output, grad_fn)
        self.watching = set()
    
    def watch(self, variable):
        self.watching.add(id(variable))
        variable._tape = self
    
    def __enter__(self):
        return self
    
    def __exit__(self, *args):
        pass
    
    def record(self, op_name, inputs, output, grad_fn):
        self.operations.append((op_name, inputs, output, grad_fn))
    
    def gradient(self, target, sources):
        """Compute gradients of target w.r.t. sources."""
        grads = {id(target): np.ones_like(target.data)}
        
        for op_name, inputs, output, grad_fn in reversed(self.operations):
            if id(output) in grads:
                input_grads = grad_fn(grads[id(output)])
                if not isinstance(input_grads, (list, tuple)):
                    input_grads = [input_grads]
                for inp, g in zip(inputs, input_grads):
                    if id(inp) not in grads:
                        grads[id(inp)] = np.zeros_like(inp.data)
                    grads[id(inp)] += g
        
        return [grads.get(id(s), np.zeros_like(s.data)) for s in sources]


class Variable:
    """Like tf.Variable - a mutable tensor."""
    def __init__(self, data, name=""):
        self.data = np.array(data, dtype=np.float64)
        self.name = name
        self._tape = None

print("GradientTape and Variable defined!")


## 2. Using GradientTape (Demo)


In [ ]:
# Compute gradient of y = x^2 at x=3
x = Variable([3.0], name="x")
w = Variable([2.0], name="w")

with GradientTape() as tape:
    tape.watch(x)
    tape.watch(w)
    
    # y = w * x^2
    x_sq = Variable(x.data ** 2)
    tape.record('square', [x], x_sq, lambda g: [g * 2 * x.data])
    
    y = Variable(w.data * x_sq.data)
    tape.record('mul', [w, x_sq], y, lambda g: [g * x_sq.data, g * w.data])

grads = tape.gradient(y, [x, w])
print(f"y = w * x^2, where w={w.data[0]}, x={x.data[0]}")
print(f"dy/dx = 2*w*x = {grads[0][0]} (expected: {2*w.data[0]*x.data[0]})")
print(f"dy/dw = x^2 = {grads[1][0]} (expected: {x.data[0]**2})")


## 3. Layer Base Class (like tf.keras.layers.Layer)


In [ ]:
class Layer:
    """Base class like tf.keras.layers.Layer."""
    
    def __init__(self, name=None):
        self.name = name or self.__class__.__name__
        self.built = False
        self._weights = []
        self.trainable = True
    
    def build(self, input_shape):
        """Create weights (called once on first input). Lazy initialization!"""
        self.built = True
    
    def call(self, inputs):
        """Forward computation (override in subclasses)."""
        raise NotImplementedError
    
    def __call__(self, inputs):
        if not self.built:
            self.build(inputs.shape)
        return self.call(inputs)
    
    def add_weight(self, shape, initializer='glorot', name=''):
        if initializer == 'glorot':
            scale = np.sqrt(2.0 / sum(shape))
            w = np.random.randn(*shape) * scale
        else:
            w = np.zeros(shape)
        self._weights.append(w)
        return len(self._weights) - 1  # return index
    
    @property
    def weights(self):
        return self._weights
    
    def get_config(self):
        return {'name': self.name, 'class': self.__class__.__name__}

print("Layer base class defined with build/call/get_config pattern!")


## 4. Dense Layer from Scratch


In [ ]:
class Dense(Layer):
    """Fully connected layer like tf.keras.layers.Dense."""
    
    def __init__(self, units, activation=None, name=None):
        super().__init__(name=name)
        self.units = units
        self.activation = activation
    
    def build(self, input_shape):
        in_features = input_shape[-1]
        self._w_idx = self.add_weight((in_features, self.units), 'glorot', 'kernel')
        self._b_idx = self.add_weight((self.units,), 'zeros', 'bias')
        self.built = True
    
    def call(self, inputs):
        w = self._weights[self._w_idx]
        b = self._weights[self._b_idx]
        output = inputs @ w + b
        if self.activation == 'relu':
            output = np.maximum(0, output)
        elif self.activation == 'sigmoid':
            output = 1 / (1 + np.exp(-np.clip(output, -500, 500)))
        elif self.activation == 'tanh':
            output = np.tanh(output)
        return output
    
    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'activation': self.activation})
        return config

# Test lazy build
layer = Dense(4, activation='relu', name='dense_1')
print(f"Before call - built: {layer.built}, weights: {len(layer.weights)}")
out = layer(np.random.randn(3, 5))  # 3 samples, 5 features
print(f"After call  - built: {layer.built}, weights: {len(layer.weights)}")
print(f"Output shape: {out.shape}")
print(f"Config: {layer.get_config()}")


## 5. Sequential Model


In [ ]:
class Sequential:
    """Like tf.keras.Sequential - a stack of layers."""
    
    def __init__(self, layers=None):
        self.layers = layers or []
        self._compiled = False
        self.optimizer = None
        self.loss_fn = None
        self.history = {'loss': [], 'accuracy': []}
    
    def add(self, layer):
        self.layers.append(layer)
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def summary(self):
        print(f"{'='*60}")
        print(f"Model: Sequential")
        print(f"{'='*60}")
        print(f"{'Layer':<20} {'Output Shape':<20} {'Params':<10}")
        print(f"{'-'*60}")
        total_params = 0
        for layer in self.layers:
            params = sum(w.size for w in layer.weights)
            total_params += params
            print(f"{layer.name:<20} {'?':<20} {params:<10}")
        print(f"{'='*60}")
        print(f"Total params: {total_params}")

model = Sequential([
    Dense(16, activation='relu', name='dense_1'),
    Dense(8, activation='relu', name='dense_2'),
    Dense(1, activation='sigmoid', name='output'),
])

# Trigger build
_ = model(np.random.randn(1, 4))
model.summary()


## 6. model.compile() - Attach Optimizer and Loss


In [ ]:
def binary_crossentropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

class Adam:
    """Adam optimizer."""
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = {}
        self.v = {}
    
    def update(self, idx, param, grad):
        if idx not in self.m:
            self.m[idx] = np.zeros_like(param)
            self.v[idx] = np.zeros_like(param)
        self.t += 1
        self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * grad
        self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * grad**2
        m_hat = self.m[idx] / (1 - self.beta1**self.t)
        v_hat = self.v[idx] / (1 - self.beta2**self.t)
        param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return param

def compile_model(model, optimizer, loss):
    model.optimizer = optimizer
    model.loss_fn = loss
    model._compiled = True
    print(f"Model compiled with {loss.__name__} loss and {optimizer.__class__.__name__} optimizer")

compile_model(model, Adam(lr=0.01), binary_crossentropy)


## 7. model.fit() - Training Loop with Callbacks


In [ ]:
class Callback:
    """Base callback class."""
    def on_epoch_end(self, epoch, logs):
        pass
    def on_train_begin(self, logs):
        pass
    def on_train_end(self, logs):
        pass

class EarlyStopping(Callback):
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.wait = 0
        self.stopped = False
    
    def on_epoch_end(self, epoch, logs):
        loss = logs.get('loss', np.inf)
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped = True
                print(f"  Early stopping at epoch {epoch}")

class PrintProgress(Callback):
    def __init__(self, every=50):
        self.every = every
    def on_epoch_end(self, epoch, logs):
        if epoch % self.every == 0:
            print(f"  Epoch {epoch:3d} - loss: {logs['loss']:.4f} - acc: {logs['accuracy']:.4f}")

print("Callbacks defined: EarlyStopping, PrintProgress")


## 8. Implementing the Full fit() Method


In [ ]:
def fit(model, X, y, epochs=100, batch_size=32, callbacks=None):
    """Training loop like model.fit() in Keras."""
    callbacks = callbacks or []
    n_samples = len(X)
    
    for cb in callbacks:
        cb.on_train_begin({})
    
    for epoch in range(epochs):
        # Shuffle
        idx = np.random.permutation(n_samples)
        X_shuffled, y_shuffled = X[idx], y[idx]
        
        epoch_losses = []
        
        for start in range(0, n_samples, batch_size):
            xb = X_shuffled[start:start+batch_size]
            yb = y_shuffled[start:start+batch_size]
            
            # Forward pass
            pred = model(xb)
            loss = model.loss_fn(yb, pred)
            epoch_losses.append(loss)
            
            # Numerical gradients (finite differences) for simplicity
            eps = 1e-5
            param_idx = 0
            for layer in model.layers:
                for i, w in enumerate(layer.weights):
                    grad = np.zeros_like(w)
                    it = np.nditer(w, flags=['multi_index'])
                    # Sample a subset of gradients for speed
                    flat_indices = np.random.choice(w.size, min(w.size, 50), replace=False)
                    for fi in flat_indices:
                        multi_idx = np.unravel_index(fi, w.shape)
                        old = w[multi_idx]
                        w[multi_idx] = old + eps
                        loss_plus = model.loss_fn(yb, model(xb))
                        w[multi_idx] = old - eps
                        loss_minus = model.loss_fn(yb, model(xb))
                        w[multi_idx] = old
                        grad[multi_idx] = (loss_plus - loss_minus) / (2 * eps)
                    
                    layer._weights[i] = model.optimizer.update(param_idx, w, grad)
                    param_idx += 1
        
        # Epoch metrics
        pred_all = model(X)
        epoch_loss = model.loss_fn(y, pred_all)
        accuracy = np.mean((pred_all > 0.5).astype(float) == y)
        
        model.history['loss'].append(epoch_loss)
        model.history['accuracy'].append(accuracy)
        
        logs = {'loss': epoch_loss, 'accuracy': accuracy}
        for cb in callbacks:
            cb.on_epoch_end(epoch, logs)
            if hasattr(cb, 'stopped') and cb.stopped:
                for cb2 in callbacks:
                    cb2.on_train_end(logs)
                return model.history
    
    for cb in callbacks:
        cb.on_train_end({})
    return model.history

print("fit() function defined!")


## 9. Training on a Classification Problem


In [ ]:
# Generate circle dataset
from numpy import pi
n = 200
angles = np.random.uniform(0, 2*pi, n)
r_inner = np.random.uniform(0, 1, n//2)
r_outer = np.random.uniform(1.5, 2.5, n//2)

X_inner = np.column_stack([r_inner * np.cos(angles[:n//2]), r_inner * np.sin(angles[:n//2])])
X_outer = np.column_stack([r_outer * np.cos(angles[n//2:]), r_outer * np.sin(angles[n//2:])])

X_data = np.vstack([X_inner, X_outer])
y_data = np.vstack([np.zeros((n//2, 1)), np.ones((n//2, 1))])

# Normalize
X_data = (X_data - X_data.mean(0)) / X_data.std(0)

print(f"Dataset: {X_data.shape[0]} samples, {X_data.shape[1]} features")
print(f"Classes: {int(y_data.sum())} positive, {int(len(y_data) - y_data.sum())} negative")


In [ ]:
# Build and train model
model = Sequential([
    Dense(16, activation='relu', name='hidden_1'),
    Dense(8, activation='relu', name='hidden_2'),
    Dense(1, activation='sigmoid', name='output'),
])
compile_model(model, Adam(lr=0.01), binary_crossentropy)

print("\nTraining...")
history = fit(model, X_data, y_data, epochs=200, batch_size=32, 
              callbacks=[PrintProgress(every=40), EarlyStopping(patience=30)])


## 10. Plot Training History


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'])
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['accuracy'])
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()


## 11. model.evaluate() and model.predict()


In [ ]:
def evaluate(model, X, y):
    """Like model.evaluate() - returns loss and metrics."""
    pred = model(X)
    loss = model.loss_fn(y, pred)
    accuracy = np.mean((pred > 0.5).astype(float) == y)
    return {'loss': loss, 'accuracy': accuracy}

def predict(model, X):
    """Like model.predict() - returns predictions."""
    return model(X)

# Evaluate
metrics = evaluate(model, X_data, y_data)
print(f"Evaluation: loss={metrics['loss']:.4f}, accuracy={metrics['accuracy']:.4f}")

# Predict on new data
X_new = np.array([[0, 0], [2, 2], [-1, -1]])
X_new_norm = (X_new - X_data.mean(0)) / X_data.std(0)
preds = predict(model, X_new_norm)
print(f"\nPredictions on new data:")
for i, (inp, p) in enumerate(zip(X_new, preds)):
    print(f"  {inp} -> {p[0]:.4f} ({'outer' if p[0] > 0.5 else 'inner'})")


## 12. The Same in Real TensorFlow/Keras

```python
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_data, y_data, epochs=200, batch_size=32,
                    callbacks=[keras.callbacks.EarlyStopping(patience=30)])

model.evaluate(X_data, y_data)
model.predict(X_new)
```

**Keras does in ~10 lines what took us ~200 lines!** But now you understand what those 10 lines do internally.


## 13. Lines of Code Comparison


In [ ]:
comparison = '''
┌─────────────────────────┬──────────────────┬─────────────────┐
│ Feature                 │ From Scratch     │ Keras           │
├─────────────────────────┼──────────────────┼─────────────────┤
│ Layer definition        │ 30 lines         │ 1 line          │
│ Model building          │ 20 lines         │ 5 lines         │
│ Compile                 │ 40 lines (Adam)  │ 1 line          │
│ Training loop           │ 60 lines         │ 1 line (fit)    │
│ Callbacks               │ 25 lines         │ 1 line          │
│ Evaluate/Predict        │ 10 lines         │ 1 line each     │
├─────────────────────────┼──────────────────┼─────────────────┤
│ TOTAL                   │ ~185 lines       │ ~10 lines       │
└─────────────────────────┴──────────────────┴─────────────────┘

But understanding those 185 lines means you TRULY understand
what Keras does when you call model.fit()!
'''
print(comparison)


## 14. tf.data Pipeline Concepts


In [ ]:
class Dataset:
    """Implements tf.data.Dataset concepts: batch, shuffle, prefetch."""
    
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self._transforms = []
    
    def shuffle(self, buffer_size):
        """Shuffle with a buffer (like tf.data)."""
        ds = Dataset(self.X, self.y)
        ds._transforms = self._transforms + [('shuffle', buffer_size)]
        return ds
    
    def batch(self, batch_size):
        ds = Dataset(self.X, self.y)
        ds._transforms = self._transforms + [('batch', batch_size)]
        return ds
    
    def prefetch(self, n=1):
        """In real TF, this overlaps data loading with training. Here conceptual."""
        ds = Dataset(self.X, self.y)
        ds._transforms = self._transforms + [('prefetch', n)]
        return ds
    
    def __iter__(self):
        X, y = self.X.copy(), self.y.copy()
        batch_size = len(X)
        
        for transform, param in self._transforms:
            if transform == 'shuffle':
                idx = np.random.permutation(len(X))
                X, y = X[idx], y[idx]
            elif transform == 'batch':
                batch_size = param
        
        for start in range(0, len(X), batch_size):
            yield X[start:start+batch_size], y[start:start+batch_size]

# Usage
ds = Dataset(X_data, y_data).shuffle(buffer_size=200).batch(32).prefetch(1)
print("tf.data-style pipeline:")
print(f"  Dataset(X, y).shuffle(200).batch(32).prefetch(1)")
print(f"\nIterating batches:")
for i, (xb, yb) in enumerate(ds):
    if i < 3:
        print(f"  Batch {i}: shapes {xb.shape}, {yb.shape}")
print(f"  ... ({i+1} total batches)")


## 15. Model Saving/Loading Concept


In [ ]:
import json as json_lib

def save_model(model, filepath):
    """Save model architecture and weights (like model.save())."""
    config = {
        'class': 'Sequential',
        'layers': [layer.get_config() for layer in model.layers]
    }
    weights = [w.tolist() for layer in model.layers for w in layer.weights]
    
    model_data = {'config': config, 'weights': weights}
    # In practice: save to file
    print(f"Model saved to '{filepath}'")
    print(f"  Architecture: {json_lib.dumps(config, indent=2)[:200]}...")
    print(f"  Weights: {len(weights)} arrays, {sum(np.array(w).size for w in weights)} parameters")
    return model_data

def load_model(model_data):
    """Reconstruct model from saved data."""
    config = model_data['config']
    print(f"Model loaded: {len(config['layers'])} layers")
    return config

saved = save_model(model, 'my_model.keras')


## 16. Summary

We built from scratch the entire Keras workflow:

| Concept | Our Implementation | TensorFlow Equivalent |
|---------|-------------------|----------------------|
| Gradient computation | `GradientTape` class | `tf.GradientTape` |
| Layer abstraction | `Layer` with build/call | `tf.keras.layers.Layer` |
| Dense layer | `Dense` class | `tf.keras.layers.Dense` |
| Model container | `Sequential` | `tf.keras.Sequential` |
| Compile pattern | `compile_model()` | `model.compile()` |
| Training loop | `fit()` function | `model.fit()` |
| Callbacks | `EarlyStopping` etc | `tf.keras.callbacks.*` |
| Data pipeline | `Dataset` class | `tf.data.Dataset` |
| Save/Load | JSON serialization | `model.save()` |

The key insight: **Keras is a carefully designed API over these exact operations.**
